<a href="https://colab.research.google.com/github/aldo02032004/naufaldo.github.io/blob/main/Linear_regression_forecast.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ✏️ ISI BAGIAN INI SAJA

In [31]:
# ==== CHANGE ONLY THIS SECTION ====
LINK_DRIVE = [   # one link, or several links separated by commas
    'https://docs.google.com/spreadsheets/d/1dYpzXZuAG7TPQjCtspagX-PqFiIh-HRG/edit',
    'https://docs.google.com/spreadsheets/d/1RBAQovVNUXinv3sgQNvDS1iMlciTL1vN/edit',
    'https://docs.google.com/spreadsheets/d/1ugTVwpxj6KQsZiI1dneQ-mAX2vTBVSTX/edit',
]
NAMA_OBJEK = 'Prabowo'   # research object name for the chart title
# ==================================

# ⚙️ Install (tidak perlu diubah)

In [32]:
import subprocess, sys
from google.colab import userdata

try:
    token = userdata.get('GH_TOKEN')
except Exception:
    raise SystemExit('❌ Secret GH_TOKEN not found. Open the 🔑 icon on the left sidebar, add GH_TOKEN, '
                     'and turn on "Notebook access".')

pkg = f'great[all] @ git+https://{token}@github.com/azmkto/GREAT-Tools.git'
result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], capture_output=True, text=True)
if result.returncode:
    print(result.stderr.replace(token, '***')[-2000:])   # never show the token in output
    raise SystemExit('❌ Install failed. Check that the token is still valid and has access to the GREAT-Tools repo.')
del token
print('✅ great library installed')

✅ great library installed


# 📈 Jalankan (tidak perlu diubah)

In [39]:
import re

import numpy as np
import pandas as pd
import ipywidgets as w
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from matplotlib.transforms import offset_copy
from IPython.display import display
from scipy.stats import t as _t_dist     # scipy always comes with scikit-learn
from sklearn.linear_model import LinearRegression

from great import sent_class, sent_colors, validate_export       # sentiment order + colors from the library
from great.viz import prep
from great.viz.overview import _panel_sentiment_trend            # same trend panel as monthly_overview
from great.viz.style import apply_style

LEVEL = 0.80         # "likely range" = 80% prediction interval: the real number lands inside about 4 days out of 5
MIN_DAYS = 2         # a trend needs at least 2 days
HORIZON_MAX = 14     # never predict more than 14 days ahead, even with long history
SMOOTH = {'None (raw daily)': 1, '7-day average': 7, '30-day average': 30}   # history display only
AUTO = 'Auto (best tested)'
SMART = 'Smart regression'
# Simplest first: when two methods score the same, Auto keeps the simpler one.
METHODS = ['Same as last day', 'Recent average', 'Same day last week', 'Straight line', 'Percent growth', SMART]
# Fewest days of data a method needs before it is offered at all.
#   Same day last week: one week to copy + one week to test it on.
#   Smart regression  : m - k_eff >= 2 with k_eff just above 3 needs m >= 6 rows, i.e. L >= 7 days.
METHOD_MIN_DAYS = {'Same day last week': 14, SMART: 7}
SMART_LAMBDAS = (1, 10, 100)      # candidate penalty strengths for the day-by-day corrections gamma_d
LAMBDA_DEFAULT = 10               # used when the backtest has fewer than 3 test cut-offs
LAMBDA_WORDS = {1: 'lightly held back', 10: 'held back', 100: 'strongly held back'}
DAY_NAMES = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']   # d = 1..7


def t_quantile(dof, level=LEVEL):
    """Student-t multiplier for a two-sided `level` range; fractional dof (Smart regression) is fine."""
    return float(_t_dist.ppf(0.5 + level / 2, dof))


class Unavailable(Exception):
    """A method that cannot be used for this setting. The message is plain English."""


def drive_xlsx_url(url):
    """Turn a Sheets (/spreadsheets/d/<id>) or Drive file (/file/d/<id>) link into an xlsx download URL."""
    m = re.search(r'/spreadsheets/d/([\w-]+)', url)
    if m:
        return f'https://docs.google.com/spreadsheets/d/{m.group(1)}/export?format=xlsx'
    m = re.search(r'/file/d/([\w-]+)', url) or re.search(r'[?&]id=([\w-]+)', url)
    if m:
        return f'https://drive.google.com/uc?export=download&id={m.group(1)}'
    raise SystemExit('❌ LINK_DRIVE is not a Google Drive / Sheets link. Copy it again from the Share button.')


def daily_counts(df1, basis):
    """Raw day x sentiment counts, empty days = 0.
    Posts = number of posts that day; People = number of unique authors that day."""
    g = df1.groupby([pd.Grouper(key='Date', freq='D'), 'Sentiment'])
    daily = g.size() if basis == 'Posts' else g['Author'].nunique()
    return (daily.unstack(fill_value=0)
                 .reindex(columns=sent_class, fill_value=0)
                 .asfreq('D', fill_value=0)
                 .astype(float))


def future_days(y, H):
    """The H calendar days right after the last day of y."""
    return pd.date_range(y.index[-1], periods=H + 1, freq='D')[1:]


def methods_for(n):
    """Methods that have enough days of data to be offered at all."""
    return [m for m in METHODS if n >= METHOD_MIN_DAYS.get(m, 0)]


# ---- Smart regression: ARX(1) on z_t = log(1 + y_t), weekend effect + ridge-penalised day corrections ----
#   z_t = c + phi * z_{t-1} + omega * W_t + sum_{d=1..7} gamma_d * D_{d,t} + eps_t,   eps_t ~ N(0, sigma^2)
#   theta = [c, phi, omega, gamma_1..gamma_7]; only gamma_d are penalised (Lambda = diag(0, 0, 0, lambda x 7)),
#   which is what makes the 7 day dummies solvable next to the intercept and W_t.
#   sklearn's Ridge would penalise every coefficient, so the penalty is added as 7 extra rows instead:
#   minimising ||z - X theta||^2 + lambda * sum gamma_d^2  ==  plain least squares on
#   X_aug = [X; sqrt(lambda) * rows that pick gamma_d],  z_aug = [z; 0]   ->  LinearRegression.

def smart_calendar(dates):
    """Calendar columns [W_t, D_{1,t}, ..., D_{7,t}]: W = Saturday/Sunday, D_d = day of week d (1 = Monday)."""
    dow = dates.dayofweek.to_numpy()                  # 0 = Monday ... 6 = Sunday, i.e. d - 1
    return np.column_stack([(dow >= 5).astype(float), (dow[:, None] == np.arange(7)).astype(float)])


_fit_cache = {}


def smart_fit(y, lookback, lam):
    """Fit Smart regression on the last L = `lookback` days of every sentiment, penalty `lam`.

    Returns {sentiment: {'theta', 'phi', 'sigma', 'dof', 'k_eff', 'fitted'}}.
    Raises Unavailable (plain English) when a rule fails. Cached per identical window."""
    win = y.iloc[-lookback:]
    key = (win.index[0], lookback, lam, win.to_numpy().tobytes())
    if key not in _fit_cache:
        try:
            _fit_cache[key] = _smart_fit(win, lam)
        except Unavailable as e:
            _fit_cache[key] = e
    result = _fit_cache[key]
    if isinstance(result, Unavailable):
        raise result
    return result


def _smart_fit(win, lam):
    L = len(win)
    m = L - 1                                         # the first day of the window has no "yesterday"
    Z = np.log1p(win.to_numpy(float))                 # z_t, t = 1..L
    cal = smart_calendar(win.index[1:])               # rows t = 2..L
    W = cal[:, 0]
    if W.all() or not W.any():
        raise Unavailable('the learned days need at least one weekday and one weekend day')
    Lam = np.diag([0.0, 0.0, 0.0] + [float(lam)] * 7)
    penalty_rows = np.sqrt(Lam[3:])                   # 7 x 10: sqrt(lambda) on gamma_1..gamma_7 only

    fits = {}
    for j, s in enumerate(win.columns):
        X = np.column_stack([np.ones(m), Z[:-1, j], cal])      # m x 10: [1, z_{t-1}, W_t, D_{1..7,t}]
        z = Z[1:, j]                                           # [z_2, ..., z_L]
        if np.linalg.matrix_rank(X[:, :3]) < 3:                # unpenalised part must be solvable on its own
            raise Unavailable(f'the {s} numbers barely changed in the learned days, so there is nothing to learn from')
        model = LinearRegression(fit_intercept=False).fit(np.vstack([X, penalty_rows]),
                                                          np.concatenate([z, np.zeros(7)]))
        theta = model.coef_                                    # theta_hat = (X'X + Lambda)^-1 X'z
        A = X.T @ X + Lam
        k_eff = float(np.trace(np.linalg.solve(A, X.T @ X)))   # trace(X (X'X + Lambda)^-1 X')
        dof = m - k_eff
        if dof < 2:
            raise Unavailable(f'it needs more days in "Days to learn from" (you have {L}, try 7 or more)')
        phi = theta[1]
        if abs(phi) >= 1:
            raise Unavailable(f'the {s} numbers keep running away in one direction in the learned days, '
                              'so the model would not settle')
        fitted = X @ theta
        fits[s] = {'theta': theta, 'phi': phi, 'k_eff': k_eff, 'dof': dof, 'fitted': fitted,
                   'sigma': np.sqrt(((z - fitted) ** 2).sum() / dof)}
    return fits


def smart_forecast(y, L, H, lam, level=LEVEL):
    """Recursive forecast on the log scale, back-transformed: median + likely range."""
    fits = smart_fit(y, L, lam)
    Cf = smart_calendar(future_days(y, H))             # [W_{n+h}, D_{1..7,n+h}] from the calendar
    Z = np.log1p(y.iloc[-L:].to_numpy(float))
    mid = np.empty((L + H, y.shape[1]))
    lo, hi = np.full_like(mid, np.nan), np.full_like(mid, np.nan)
    for j, s in enumerate(y.columns):
        f = fits[s]
        c, phi, rest = f['theta'][0], f['phi'], f['theta'][2:]
        cal = Cf @ rest                                # omega * W_{n+h} + sum_d gamma_d * D_{d,n+h}
        z, zf = Z[-1, j], np.empty(H)                  # z_hat_n = z_n
        for h in range(H):
            z = c + phi * z + cal[h]                   # z_hat_{n+h}
            zf[h] = z
        se = f['sigma'] * np.sqrt(np.cumsum(phi ** (2 * np.arange(H))))    # SE_h
        q = t_quantile(f['dof'], level)
        mid[0, j] = Z[0, j]                            # first learned day has no fit: show the real value
        mid[1:L, j] = f['fitted']
        mid[L:, j] = zf
        lo[L:, j], hi[L:, j] = zf - q * se, zf + q * se
    return np.expm1(mid), np.expm1(lo), np.expm1(hi)


def forecast(y, lookback, horizon, method, level=LEVEL, lam=LAMBDA_DEFAULT):
    """Predict horizon days after the end of y, using the last lookback days.

    Returns (mid, lower, upper) indexed over the lookback days + the predicted days.
    lower/upper = likely range at `level` (NaN where it cannot be estimated from so few days).
      Straight line    : y = a*t + b, OLS prediction interval (widens further ahead)
      Percent growth   : same on log(1 + y), so change is a % per day and never below 0
      Recent average   : flat line at the mean of the lookback days
      Same as last day / Same day last week : copy recent values (range: random walk / none)
      Smart regression : yesterday + weekend + penalised day-of-week on log(1 + y), penalty `lam`;
                         raises Unavailable when it cannot be fitted"""
    L, H = lookback, horizon
    Y = y.iloc[-L:].to_numpy(float)
    mid = np.full((L + H, y.shape[1]), np.nan)
    lo, hi = mid.copy(), mid.copy()
    steps = np.arange(1, H + 1, dtype=float)[:, None]          # 1..H days ahead

    if method in ('Straight line', 'Percent growth'):
        Yf = np.log1p(Y) if method == 'Percent growth' else Y
        t0 = np.arange(L, dtype=float)
        model = LinearRegression().fit(t0[:, None], Yf)          # one line per sentiment: y = a*t + b
        a, b = model.coef_[:, 0], model.intercept_
        t = np.arange(L + H, dtype=float)[:, None]
        mid = a * t + b
        if L >= 3:
            s = np.sqrt(((Yf - (a * t0[:, None] + b)) ** 2).sum(axis=0) / (L - 2))
            se = s * np.sqrt(1 + 1 / L + (t - t0.mean()) ** 2 / ((t0 - t0.mean()) ** 2).sum())
            q = t_quantile(L - 2, level)
            lo, hi = mid - q * se, mid + q * se
        if method == 'Percent growth':
            mid, lo, hi = np.expm1(mid), np.expm1(lo), np.expm1(hi)

    elif method == 'Recent average':
        m = Y.mean(axis=0)
        mid[:] = m
        if L >= 2:
            se = Y.std(axis=0, ddof=1) * np.sqrt(1 + 1 / L)
            q = t_quantile(L - 1, level)
            lo[L:], hi[L:] = m - q * se, m + q * se

    elif method == 'Same as last day':
        mid[L - 1:] = Y[-1]
        if L >= 3:
            se = np.diff(Y, axis=0).std(axis=0, ddof=1) * np.sqrt(steps)
            q = t_quantile(L - 2, level)
            lo[L:], hi[L:] = Y[-1] - q * se, Y[-1] + q * se

    elif method == 'Same day last week':
        mid[L - 1] = Y[-1]
        mid[L:] = y.iloc[-7:].to_numpy(float)[np.arange(H) % 7]

    elif method == SMART:
        mid, lo, hi = smart_forecast(y, L, H, lam, level)

    else:
        raise ValueError(method)

    idx = y.index[-L:].append(future_days(y, H))
    frame = lambda v: pd.DataFrame(np.clip(v, 0, None), index=idx, columns=y.columns)
    return frame(mid), frame(lo), frame(hi)


def backtest(y, lookback, horizon, methods):
    """Replay the past: pretend each earlier day is 'today', predict the next horizon days
    with every method, and compare with what really happened. All methods use the same test days.

    Smart regression is scored once per candidate lambda on those same test days and the most
    accurate lambda is kept (lambda = 10 when there are fewer than 3 test days).
    NOTE: lambda is chosen on the same backtest that reports its accuracy, so the Smart
    regression score is slightly optimistic.
    A method (or lambda) that cannot be fitted on any test day is left out (see 'skipped').

    accuracy = 1 - total miss / total real value  (100% = perfect, 0% = missed by 100% or more)
    coverage = share of real values that landed inside the likely range"""
    variants = [(m, lam) for m in methods for lam in (SMART_LAMBDAS if m == SMART else (LAMBDA_DEFAULT,))]
    start = max(lookback, 7) if 'Same day last week' in methods else lookback
    miss = {v: 0.0 for v in variants}
    inside = {v: [] for v in variants}
    skipped = {}
    real_total, tests = 0.0, 0
    for end in range(start, len(y) - horizon + 1):
        hist, actual = y.iloc[:end], y.iloc[end:end + horizon].to_numpy()
        real_total += actual.sum()
        tests += 1
        for v in variants:
            if v in skipped:
                continue
            m, lam = v
            try:
                mid, lo, hi = (f.iloc[lookback:].to_numpy() for f in forecast(hist, lookback, horizon, m, lam=lam))
            except Unavailable as e:
                skipped[v] = f'on {hist.index[-1]:%d %b %Y} {e}'
                continue
            miss[v] += np.abs(mid - actual).sum()
            if not np.isnan(lo).all():
                inside[v].append(((actual >= lo) & (actual <= hi)).mean())
    if tests == 0 or real_total == 0:
        return None

    acc = {v: max(0.0, 1 - miss[v] / real_total) for v in variants if v not in skipped}
    smart_lambda = LAMBDA_DEFAULT
    fitted_lams = [lam for lam in SMART_LAMBDAS if (SMART, lam) in acc]
    if SMART in methods and tests >= 3 and fitted_lams:
        smart_lambda = max(fitted_lams, key=lambda lam: acc[(SMART, lam)])
    chosen = {m: (m, smart_lambda if m == SMART else LAMBDA_DEFAULT) for m in methods}
    return {'tests': tests, 'smart_lambda': smart_lambda,
            'accuracy': {m: acc[v] for m, v in chosen.items() if v in acc},
            'coverage': {m: (float(np.mean(inside[v])) if inside[v] else None)
                         for m, v in chosen.items() if v in acc},
            'skipped': {m: skipped[v] for m, v in chosen.items() if v in skipped}}


# ---- 1. Load data from Drive (one link or several) ----
links = [x.strip() for x in ([LINK_DRIVE] if isinstance(LINK_DRIVE, str) else LINK_DRIVE) if x.strip()]
if not links:
    raise SystemExit('❌ LINK_DRIVE is empty. Paste at least one Google Drive / Sheets link in the first cell.')

frames = []
for i, link in enumerate(links, 1):
    label = f'link {i} of {len(links)}' if len(links) > 1 else 'the link'
    try:
        part = pd.read_excel(drive_xlsx_url(link), header=1)
    except SystemExit:
        raise
    except Exception as e:
        raise SystemExit(f'❌ Could not open {label} ({e}).\n'
                         'Make sure the Drive file is shared as "Anyone with the link".')
    try:
        frames.append(validate_export(part))
    except ValueError as e:
        raise SystemExit(f'❌ The file in {label} does not match the export format: {e}')
    if len(links) > 1:
        print(f'   {label}: {len(part):,} rows')

df = pd.concat(frames, ignore_index=True)
# Files covering overlapping days hold the same posts twice. 'No' is just a row number per file, so ignore it.
df['Date'] = pd.to_datetime(df['Date'], format='ISO8601')    # one type for every file, so duplicates match
before = len(df)
df = df.drop_duplicates(subset=[c for c in df.columns if c != 'No']).reset_index(drop=True)
if len(df) < before:
    print(f'   removed {before - len(df):,} duplicate rows that were in more than one file')

# Read the time of the last post BEFORE prepare_data(), which keeps only the date.
last_post = df['Date'].max()
df1, start_date, end_date = prep.prepare_data(df)
n_days = (df1['Date'].max() - df1['Date'].min()).days + 1
print(f'✅ {len(df1):,} rows loaded | {start_date} - {end_date} | n = {n_days} days')
if n_days < MIN_DAYS:
    raise SystemExit(f'❌ Only {n_days} day of data. A trend needs at least {MIN_DAYS} days.')

# If the export was pulled mid-day, the last day is incomplete and drags the trend down.
has_time = (df['Date'] != df['Date'].dt.normalize()).any()
partial_guess = bool(has_time and last_post.hour < 23 and n_days > MIN_DAYS)
if partial_guess:
    print(f'⚠️ Last post is at {last_post:%H:%M}. The last day looks incomplete and is skipped by default.')

_cache = {}


def cached(key, make):
    """Compute `make()` once per key, so moving a slider back and forth stays fast."""
    if key not in _cache:
        _cache[key] = make()
    return _cache[key]


def get_daily(basis_v, drop_v):
    d = cached(('daily', basis_v), lambda: daily_counts(df1, basis_v))
    return d.iloc[:-1] if drop_v else d


def get_backtest(y, basis_v, drop_v, learn_v, predict_v):
    return cached(('bt', basis_v, drop_v, learn_v, predict_v),
                  lambda: backtest(y, learn_v, predict_v, methods_for(len(y))))


def get_forecast(y, basis_v, drop_v, learn_v, predict_v, used, lam):
    return cached(('fc', basis_v, drop_v, learn_v, predict_v, used, lam),
                  lambda: forecast(y, learn_v, predict_v, used, lam=lam))


# ---- 2. Interface ----
apply_style(dpi=100)   # library style; lower dpi so the chart redraws quickly
wide = {'description_width': '140px'}
usable = n_days - int(partial_guess)                     # days left after skipping an incomplete last day

# The two choices a new user needs.
basis   = w.ToggleButtons(options=['Posts', 'People'], description='What to count', style=wide,
                          tooltips=['Number of posts per day', 'Number of different people posting per day'])
predict = w.IntSlider(min=1, max=min(HORIZON_MAX, usable), value=min(3, usable),
                      description='Days to predict', style=wide)

# Everything else keeps a sensible default and sits in a closed "Advanced settings" box.
method    = w.Dropdown(options=[AUTO] + methods_for(n_days), value=AUTO, description='Method', style=wide)
learn     = w.IntSlider(min=MIN_DAYS, max=usable, value=min(7, usable),
                        description='Days to learn from', style=wide)
smooth    = w.Dropdown(options=[k for k, v in SMOOTH.items() if v <= n_days], description='Smooth the line',
                       style=wide)
drop_last = w.Checkbox(value=partial_guess, description='Ignore the last day (not complete yet)',
                       indent=False, disabled=n_days <= MIN_DAYS)


def _update_limits(change=None):
    """Both sliders depend on how many days are available."""
    n = n_days - int(drop_last.value)
    learn.max = n
    predict.max = min(HORIZON_MAX, n)


drop_last.observe(_update_limits, names='value')

intro = w.HTML(
    '<b>How to use:</b> choose what to count and how many days to predict. That is all.<br>'
    f'Dashed line = prediction. Shaded area = likely range (the real number lands inside it about {LEVEL:.0%} '
    'of the time). How accurate it was on past days is written under the chart.')

advanced_help = w.HTML(
    '<b>Method</b>: leave it on Auto. Auto tests every method on past days and uses the most accurate one.<br>'
    '<b>Days to learn from</b>: how many recent days the prediction looks at. More = calmer, fewer = reacts faster.<br>'
    '<b>Smooth the line</b>: only makes the past line easier to read. The prediction always uses the real daily numbers.<br>'
    '<b>Ignore the last day</b>: tick it when the data was downloaded in the middle of the day.<br>'
    '<i>Methods: Same as last day · Recent average (flat line) · Same day last week (needs 14+ days) · '
    'Straight line · Percent growth · Smart regression (learns from yesterday and from weekends vs weekdays)</i>')

advanced = w.Accordion(children=[w.VBox([w.HBox([method, smooth]), w.HBox([learn, drop_last]), advanced_help])],
                       selected_index=None)                  # closed until the user opens it
advanced.set_title(0, 'Advanced settings (optional)')


# ---- 3. Chart + accuracy text ----
def smart_text(y, learn_v, lam):
    """Plain-English summary of Smart regression, for the sentiment with the most volume."""
    try:
        fits = smart_fit(y, learn_v, lam)
    except Unavailable as e:
        return ['', f'Smart regression: cannot be used with this setting, because {e}.']

    big = y.iloc[-learn_v:].sum().idxmax()
    f = fits[big]
    phi, omega, gamma = f['phi'], f['theta'][2], f['theta'][3:]
    carry = (f'a spike fades by half in about {np.log(0.5) / np.log(phi):.1f} days' if 0 < phi < 1
             else 'no carry-over from yesterday')
    wk = np.expm1(omega) * 100
    weekend = f'weekends are about {abs(wk):.0f}% {"higher" if wk > 0 else "lower"} than weekdays'
    pct = np.expm1(gamma) * 100
    if np.all(np.abs(pct) < 2):
        detail = 'no clear day-by-day differences'
    else:
        d = int(np.argmax(np.abs(gamma)))
        detail = (f'{DAY_NAMES[d]} about {abs(pct[d]):.0f}% {"above" if pct[d] > 0 else "below"} '
                  f'other {"weekend days" if d >= 5 else "weekdays"}')
    return ['', f'Smart regression parts (shown for "{big}", the biggest sentiment):',
            f'  Yesterday effect ... {carry}',
            f'  Weekend effect ..... {weekend}',
            f'  Day-by-day detail .. {LAMBDA_WORDS[lam]} ({detail})']


def accuracy_text(bt, used, picked_auto, n, learn_v, predict_v, y, lam, note=None):
    lines = ['HOW ACCURATE IS THIS PREDICTION?']
    if note:
        lines.append(note)
    if bt is None:
        lines += ['Not enough past days to test this setting.',
                  'Lower "Days to learn from" or "Days to predict" to get an accuracy score.']
        if picked_auto:
            lines.append(f'Auto used "{used}": usually the safest choice when there is very little data.')
    else:
        acc = bt['accuracy']
        lines += [f'Test: pretend an earlier day is "today", predict the next {predict_v} day(s), '
                  f'compare with what really happened. Repeated {bt["tests"]}x.',
                  'Accuracy = 100% minus the average miss, as a % of the real numbers.', '']
        for m in sorted(acc, key=acc.get, reverse=True):
            tag = '  <- used' + (' (picked by Auto)' if picked_auto else '') if m == used else ''
            lines.append(f'  {m:<20} {acc[m]:>4.0%}{tag}')
        for m, why in bt['skipped'].items():
            lines.append(f'  {m:<20}  n/a  (not tested: {why})')
        best = max(acc, key=acc.get)
        if used != best:
            lines += ['', f'Tip: "{best}" scored better for this setting.']
        cov = bt['coverage'].get(used)
        if cov is not None:
            lines += ['', f'Shaded range caught the real value {cov:.0%} of the time (target {LEVEL:.0%}).']
        if bt['tests'] < 5:
            lines.append(f'⚠️ Only {bt["tests"]} test(s) possible, so these scores are rough. More days of data = more reliable score.')

    if SMART in methods_for(n):
        lines += smart_text(y, learn_v, lam)

    if n < 7:
        lines.append('⚠️ Less than one week of data: weekday vs weekend differences are mixed in. Treat as rough.')
    if learn_v < 3:
        lines.append('⚠️ Learning from only 2 days: no likely range can be calculated. Use 3 days or more.')
    if predict_v > learn_v:
        lines.append('⚠️ Predicting further ahead than the days it learned from. Expect bigger misses.')
    return '\n'.join(lines)


def draw(basis_v, method_v, smooth_v, learn_v, predict_v, drop_v):
    y = get_daily(basis_v, drop_v)
    n = len(y)
    if n < MIN_DAYS:
        print(f'❌ Only {n} day left. Untick "Ignore the last day" in Advanced settings.')
        return
    learn_v, predict_v = min(learn_v, n), min(predict_v, n)
    if method_v not in methods_for(n):
        method_v = AUTO

    bt = get_backtest(y, basis_v, drop_v, learn_v, predict_v)
    lam = bt['smart_lambda'] if bt else LAMBDA_DEFAULT
    picked_auto = method_v == AUTO
    if picked_auto:
        used = max(bt['accuracy'], key=bt['accuracy'].get) if bt else 'Recent average'
    else:
        used = method_v

    note = None
    try:
        mid, lo, hi = get_forecast(y, basis_v, drop_v, learn_v, predict_v, used, lam)
    except Unavailable as e:
        note = f'"{used}" cannot be used with this setting, because {e}. Showing "Recent average" instead.'
        used = 'Recent average'
        mid, lo, hi = get_forecast(y, basis_v, drop_v, learn_v, predict_v, used, lam)
    fc, fc_lo, fc_hi = (f.iloc[learn_v:] for f in (mid, lo, hi))
    shown = y.rolling(SMOOTH[smooth_v], min_periods=1).mean()      # display only

    fig, ax = plt.subplots(figsize=(30, 8))
    _panel_sentiment_trend(ax, shown, sent_class, sent_colors,
                           f'{NAMA_OBJEK} ({basis_v}, {smooth_v}) | prediction: {used}',
                           f'{y.index[0]:%d %b %Y}', f'{y.index[-1]:%d %b %Y}', interval=1)   # days actually shown

    # The dashed line starts at the last point of the history line, so the two lines join up.
    # The likely range covers the predicted days only (no funnel from the last real day).
    line = pd.concat([shown.iloc[[-1]], fc])
    for s in sent_class:
        c = sent_colors[s]
        ax.plot(line.index, line[s], color=c, linewidth=3, linestyle='--', marker='o', ms=3.01, alpha=0.8)
        ax.fill_between(fc.index, fc_lo[s], fc_hi[s], color=c, alpha=0.12, linewidth=0)

    # End-of-line labels, lowest first; a label that would sit on top of the one below is pushed up.
    ymin, ymax = ax.get_ylim()
    gap, prev = 0.05 * (ymax - ymin), -np.inf
    x_end = mdates.date2num(fc.index[-1])
    shift = offset_copy(ax.transData, fig=fig, x=8, units='points')
    for s in sorted(sent_class, key=lambda s: fc[s].iloc[-1]):
        last, low, high = fc[s].iloc[-1], fc_lo[s].iloc[-1], fc_hi[s].iloc[-1]
        text = f'{last:,.0f}' if np.isnan(low) else f'{last:,.0f} ({low:,.0f}–{high:,.0f})'
        prev = max(last, prev + gap)
        ax.annotate(text, (x_end, last), xytext=(x_end, prev), textcoords=shift,
                    va='center', fontsize=12, color=sent_colors[s], fontweight='bold')
    ax.axvspan(y.index[-learn_v], y.index[-1], color='orange', alpha=0.12,
               label=f'Learned from ({learn_v} days)')

    all_idx = y.index.append(fc.index)
    ax.set_xticks(all_idx[::max(1, len(all_idx) // 15)])   # at most ~15 date labels
    unit = 'Posts' if basis_v == 'Posts' else 'People'
    ax.set_ylabel(f'{unit} per day' + ('' if SMOOTH[smooth_v] == 1 else f' ({smooth_v})'),
                  fontsize=18, fontweight='semibold')
    ax.legend(fontsize=12, loc='upper center', bbox_to_anchor=(0.5, 1.05), frameon=False, ncol=5)
    plt.tight_layout()
    plt.show()

    print(accuracy_text(bt, used, picked_auto, n, learn_v, predict_v, y, lam, note))


out = w.interactive_output(draw, {'basis_v': basis, 'method_v': method, 'smooth_v': smooth,
                                  'learn_v': learn, 'predict_v': predict, 'drop_v': drop_last})
display(w.VBox([intro, w.HBox([basis, predict]), advanced, out]))

   removed 26,926 duplicate rows that were in more than one file
✅ 123,074 rows loaded | 11 Sep 2026 - 25 Sep 2026 | n = 15 days
⚠️ Last post is at 13:46. The last day looks incomplete and is skipped by default.
